In [ ]:
!pip install -U bitsandbytes transformers accelerate datasets spacy nltk
!python -m spacy download en_core_web_sm
!pip install torch
!pip install -U bitsandbytes>=0.46.1

In [ ]:
# =============================================================================
# BLOCK 1: IMPORTS AND MODEL LOADING (4-BIT QUANTIZATION)
# =============================================================================
import os
import torch
import json
import random
import numpy as np
import spacy
import nltk
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from nltk.tokenize import sent_tokenize
import gc

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("="*80)
print("BLOCK 1: MODEL LOADING")
print("="*80)

# Configuration
# ============================================================
# CHANGED: Use Mistral-7B instead of Falcon (better compatibility)
# ============================================================
MODEL_NAME = "mistralai/Mistral-7B-v0.1"  # ← CHANGE THIS LINE

TOPK_FIRST_TOKEN = 4
WINDOWS = 16
TARGET_SAMPLES = 1000  # Per class

# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

print(f"\nLoading {MODEL_NAME} with 4-bit quantization...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Some models don't have pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

model.eval()

print(f"✓ Model loaded on {torch.cuda.get_device_name(0)}")
print(f"✓ Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
print(f"✓ Hidden dimension: {model.config.hidden_size}")

# Load NLP
nlp = spacy.load("en_core_web_sm")
print("✓ Spacy loaded")

In [ ]:
# =============================================================================
# BLOCK 2: ENTITY EXTRACTION (EXACT FROM SHARED CODE)
# =============================================================================
print("\n" + "="*80)
print("BLOCK 2: ENTITY EXTRACTION FUNCTIONS")
print("="*80)

def delete_substrings(lst):
    """Remove strings that are substrings of others"""
    substrings = []
    lst = list(set(lst))
    for s in lst:
        if any(s in o for o in lst if o != s):
            substrings.append(s)
    for s in substrings:
        lst.remove(s)
    return lst

def find_boundaries(text, words):
    """Expand entities to word boundaries"""
    boundaries = []
    for word in words:
        ntext = text
        while True:
            start = ntext.find(word)
            if start == -1:
                break
            end = start + len(word) - 1
            while start > 0 and ntext[start-1] != " ":
                start -= 1
            while end < len(ntext)-1 and ntext[end+1] != " ":
                end += 1
            boundaries.append("".join([ntext[i] for i in range(start, end+1)]))
            ntext = ntext[end+1:]
    return boundaries

def get_entities(text):
    """Extract all entity occurrences (EXACT from shared code)"""
    entities_ = list(set([str(e) for e in nlp(text).ents]))
    entities_ = find_boundaries(text, entities_)
    entities = delete_substrings(entities_)
    all_entities = []
    for i in range(len(text)):
        for e in entities:
            if text[i:].startswith(e):
                all_entities.append((e, i))
    return all_entities

print("✓ Entity extraction functions defined")

In [ ]:
# =============================================================================
# BLOCK 3: TOKEN-LEVEL GENERATION (EXACT FROM SHARED CODE)
# =============================================================================
print("\n" + "="*80)
print("BLOCK 3: TOKEN-LEVEL GENERATION")
print("="*80)

def find_first_and_next_token(text, e, idx, input_id, prompt=""):
    """Find first_token and next_token (EXACT from shared code)"""
    new_text = f"{text[:idx].strip()} {text[idx:].replace(e, e + ' @', 1).strip()}"
    new_input_id = tokenizer(prompt + new_text.strip(), return_tensors='pt')['input_ids'].tolist()[0]

    # Verify prompt matches
    for i in range(len(input_id[0])):
        if input_id[0][i] != new_input_id[i]:
            return []

    first_token = new_input_id[len(input_id[0])]

    # Find @ token
    at_token_candidates = tokenizer("@", add_special_tokens=False)['input_ids']
    at_token_candidates += tokenizer(" @", add_special_tokens=False)['input_ids']

    at_position = None
    for at_tok in at_token_candidates:
        try:
            at_position = new_input_id.index(at_tok, len(input_id[0]))
            break
        except ValueError:
            continue

    if at_position is None or at_position >= len(new_input_id) - 1:
        return []

    next_token = new_input_id[at_position + 1]
    entity_len = at_position - len(input_id[0])
    last_id = new_input_id[at_position + 1:]

    return [first_token, next_token, entity_len, last_id]

print("✓ Token detection function defined")

In [ ]:
# =============================================================================
# BLOCK 4: EMBEDDING EXTRACTION (EXACT FROM SHARED CODE - get_hd function)
# =============================================================================
print("\n" + "="*80)
print("BLOCK 4: EMBEDDING EXTRACTION")
print("="*80)

def get_hd(text):
    """
    Extract hidden states EXACTLY as in shared code (get_hd function)

    Returns 3 embeddings:
    1. hds: Average of all layers' last token
    2. hds_mean_1: Mean of first layer tokens (from position 2 onwards)
    3. hds_mean_2: Mean of last layer tokens (from position 2 onwards)
    """
    # Tokenize
    ids = tokenizer(text.strip(), return_tensors='pt')['input_ids'].tolist()

    # Get hidden states
    with torch.no_grad():
        output = model(torch.tensor(ids).to(model.device), output_hidden_states=True)

    hd = output.hidden_states

    # Method 1: Average of all layers' last token
    hds = hd[1][0][-1].clone().detach()
    for i in range(2, len(hd)):
        hds += hd[i][0][-1].clone().detach()
    hds = hds / (len(hd) - 1)

    # Start position (for base models, start at token 2)
    start_at = 2

    # Method 2: Mean of first layer from start_at onwards
    hds_mean_1 = torch.mean(hd[1][0][start_at-1:], dim=0)

    # Method 3: Mean of last layer from start_at onwards
    hds_mean_2 = torch.mean(hd[-1][0][start_at-1:], dim=0)

    return hds.tolist(), hds_mean_1.tolist(), hds_mean_2.tolist()

print("✓ Embedding extraction function defined (3 methods)")
print("  1. hds: Average of all layers' last token")
print("  2. hds_mean_1: Mean of first layer")
print("  3. hds_mean_2: Mean of last layer (MIND paper standard)")

In [ ]:
# =============================================================================
# BLOCK 5: TEST ON SAMPLE TEXT (WITH ATTENTION MASK FIX)
# =============================================================================
print("\n" + "="*80)
print("BLOCK 5: TESTING ON SAMPLE TEXT")
print("="*80)

sample_text = """Albert Einstein was born in Ulm, in the Kingdom of Württemberg in the German Empire, on 14 March 1879. His parents were Hermann Einstein, a salesman and engineer, and Pauline Koch."""

print(f"\nSample text: {sample_text[:100]}...")

entities = get_entities(sample_text)
print(f"✓ Found {len(entities)} entity occurrences")

# Filter
title = "Albert Einstein"
valid_entities = [(e, idx) for e, idx in entities if idx != 0 and e not in title]
print(f"✓ Valid entities: {len(valid_entities)}")

if valid_entities:
    entity, char_idx = valid_entities[0]
    print(f"\n✓ Testing with entity: '{entity}' at position {char_idx}")

    prompt = sample_text[:char_idx].strip()
    print(f"✓ Prompt: {prompt[:60]}...")

    # ============================================================
    # FIX: Create proper input_ids with attention_mask
    # ============================================================
    encoded_input = tokenizer(prompt, return_tensors='pt')
    input_ids = encoded_input['input_ids'].to(model.device)
    attention_mask = encoded_input['attention_mask'].to(model.device)
    input_id = input_ids.tolist()  # For compatibility with existing code

    tokens = find_first_and_next_token(sample_text, entity, char_idx, input_id)

    if tokens:
        first_, next_, entity_len, last_id = tokens
        print(f"✓ first_token={first_}, next_token={next_}, entity_len={entity_len}")

        # ============================================================
        # FIX: Generate with attention_mask
        # ============================================================
        output = model.generate(
            input_ids,
            attention_mask=attention_mask,  # ← ADDED
            max_new_tokens=1,
            return_dict_in_generate=True,
            output_scores=True,
            pad_token_id=tokenizer.eos_token_id,
        )

        values, indices = torch.topk(output.scores[0], k=TOPK_FIRST_TOKEN)
        top_tokens = indices[0].tolist()

        if first_ in top_tokens:
            print(f"✗ Model predicts correct entity (skip)")
        else:
            print(f"✓ Model does NOT predict entity (hallucination candidate)")

            # Search for next_token
            sequences = output.sequences
            found = False

            for i in range(entity_len + WINDOWS):
                # ============================================================
                # FIX: Update attention_mask for each new token
                # ============================================================
                attention_mask = torch.ones_like(sequences)  # ← ADDED

                output = model.generate(
                    sequences,
                    attention_mask=attention_mask,  # ← ADDED
                    max_new_tokens=1,
                    return_dict_in_generate=True,
                    output_scores=True,
                    pad_token_id=tokenizer.eos_token_id,
                )
                values, indices = torch.topk(output.scores[0], k=TOPK_FIRST_TOKEN)
                if next_ in indices[0].tolist():
                    print(f"  ✓ Found next_token at step {i+1}")
                    found = True
                    break
                sequences = output.sequences

            if found:
                new_entity_id = sequences[0].tolist()[len(input_id[0]):]
                all_new_text_id = input_id[0] + new_entity_id + last_id
                hallucinated_text = tokenizer.decode(all_new_text_id, skip_special_tokens=True)

                print(f"\n✓ HALLUCINATED TEXT GENERATED:")
                print(f"  {hallucinated_text[:150]}...")

                # Extract embeddings for both
                hds_hall, mean1_hall, mean2_hall = get_hd(hallucinated_text)
                hds_orig, mean1_orig, mean2_orig = get_hd(sample_text)

                print(f"\n✓ Embeddings extracted:")
                print(f"  Hallucinated - dim={len(mean2_hall)}")
                print(f"  Original - dim={len(mean2_orig)}")
            else:
                print("✗ next_token not found within window")
    else:
        print("✗ Token detection failed")

print("\n" + "="*80)
print("BLOCK 5 COMPLETE")
print("="*80)

In [ ]:
"""
=============================================================================
BLOCK 6: DATASET GENERATION (EXACT SHARED CODE LOGIC)
=============================================================================
"""

print("\n" + "="*80)
print("BLOCK 6: DATASET GENERATION")
print("="*80)

def generate_sample(text, entity, idx, title):
    """
    Generate sample following EXACT shared code logic:
    1. Token-level generation
    2. Extract embeddings using get_hd()
    3. Return both hallucinated and non-hallucinated samples
    """
    # Truncate at entity
    input_id = tokenizer(text[:idx].strip(), return_tensors='pt')['input_ids'].tolist()

    # Find tokens
    tokens = find_first_and_next_token(text, entity, idx, input_id)
    if not tokens:
        return None

    first_, next_, entity_len, last_id = tokens

    # Stage A: Generate one token
    output = model.generate(
        torch.tensor(input_id).to(model.device),
        max_new_tokens=1,
        return_dict_in_generate=True,
        output_scores=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    # Check if model predicts correct entity
    values, indices = torch.topk(output.scores[0], k=TOPK_FIRST_TOKEN)
    if first_ in indices[0].tolist():
        return None  # Model knows answer, skip

    # Stage C: Search for next_token
    sequences = output.sequences
    found = False

    for i in range(entity_len + WINDOWS):
        output = model.generate(
            sequences,
            max_new_tokens=1,
            return_dict_in_generate=True,
            output_scores=True,
            pad_token_id=tokenizer.eos_token_id,
        )
        values, indices = torch.topk(output.scores[0], k=TOPK_FIRST_TOKEN)
        if next_ in indices[0].tolist():
            found = True
            break
        sequences = output.sequences

    if not found:
        return None

    # Build hallucinated text
    new_sequence = sequences[0].tolist()
    new_entity_id = new_sequence[len(input_id[0]):]
    all_new_text_id = input_id[0] + new_entity_id + last_id
    hallucinated_text = tokenizer.decode(all_new_text_id, skip_special_tokens=True)

    # Extract new entity
    # Find @ markers
    temp_ids = input_id[0] + new_entity_id + [tokenizer.encode("@", add_special_tokens=False)[0]]
    temp_text = tokenizer.decode(temp_ids, skip_special_tokens=True)
    new_entity = tokenizer.decode(new_entity_id, skip_special_tokens=True).strip().lower()

    # Quality check
    if (not new_entity or
        entity.lower() in new_entity or
        any(ee.strip() in text.lower() for ee in new_entity.split(" ") if ee.strip())):
        return None

    # Extract embeddings using get_hd() - EXACT from shared code
    # For hallucinated text
    hds_hall, mean1_hall, mean2_hall = get_hd(hallucinated_text)

    # For original text (non-hallucinated)
    hds_orig, mean1_orig, mean2_orig = get_hd(text)

    return {
        # Hallucinated sample (label=1)
        'text_hall': hallucinated_text,
        'entity_hall': new_entity,
        'hds_hall': hds_hall,           # Average of all layers' last token
        'mean1_hall': mean1_hall,       # Mean of first layer
        'mean2_hall': mean2_hall,       # Mean of last layer
        'label_hall': 1,

        # Non-hallucinated sample (label=0)
        'text_orig': text,
        'entity_orig': entity,
        'hds_orig': hds_orig,
        'mean1_orig': mean1_orig,
        'mean2_orig': mean2_orig,
        'label_orig': 0,

        # Metadata
        'title': title,
    }


# Load Wikipedia
print(f"\nLoading Wikipedia dataset...")
wiki = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)

# Generation loop
dataset_hall = []
dataset_non_hall = []
processed = 0

print(f"\nGenerating {TARGET_SAMPLES*2} samples ({TARGET_SAMPLES} per class)...")
print("This will take 1-2 hours...\n")

pbar = tqdm(total=TARGET_SAMPLES*2, desc="Samples")

for row in wiki:
    if len(dataset_hall) >= TARGET_SAMPLES and len(dataset_non_hall) >= TARGET_SAMPLES:
        break

    processed += 1

    try:
        sents = sent_tokenize(row['text'])
        if len(sents) < 2:
            continue

        text = " ".join(sents[:2])
        title = row.get('title', '')

        entities = get_entities(text)
        if not entities:
            continue

        # Filter and deduplicate
        entities_filtered = []
        idx_seen = []
        for e, idx in entities:
            if idx == 0 or e in title:
                continue
            if idx not in idx_seen:
                idx_seen.append(idx)
                entities_filtered.append((e, idx))

        if not entities_filtered:
            continue

        # Try one entity
        entity, char_idx = random.choice(entities_filtered)
        result = generate_sample(text, entity, char_idx, title)

        if result is None:
            continue

        # Add hallucinated sample
        if len(dataset_hall) < TARGET_SAMPLES:
            dataset_hall.append({
                'label': result['label_hall'],
                'text': result['text_hall'],
                'entity': result['entity_hall'],
                'embedding': result['mean2_hall'],  # Use mean of last layer (MIND standard)
                'title': result['title'],
            })
            pbar.update(1)

        # Add non-hallucinated sample
        if len(dataset_non_hall) < TARGET_SAMPLES:
            dataset_non_hall.append({
                'label': result['label_orig'],
                'text': result['text_orig'],
                'entity': result['entity_orig'],
                'embedding': result['mean2_orig'],  # Use mean of last layer
                'title': result['title'],
            })
            pbar.update(1)

        pbar.set_postfix({
            'H=1': len(dataset_hall),
            'H=0': len(dataset_non_hall),
            'Proc': processed
        })

        # Checkpoint every 500
        if (len(dataset_hall) + len(dataset_non_hall)) % 500 == 0:
            with open('llama32_checkpoint.json', 'w') as f:
                json.dump(dataset_hall + dataset_non_hall, f)

    except Exception as e:
        if processed % 1000 == 0:
            print(f"\n[Warn at {processed}]: {e}")
        continue

    if processed % 100 == 0:
        torch.cuda.empty_cache()

pbar.close()

print(f"\n✓ Generation complete!")
print(f"  Hallucinated (label=1): {len(dataset_hall)}")
print(f"  Non-hallucinated (label=0): {len(dataset_non_hall)}")
print(f"  Total: {len(dataset_hall) + len(dataset_non_hall)}")




BLOCK 6: DATASET GENERATION

Loading Wikipedia dataset...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]


Generating 3600 samples (1800 per class)...
This will take 1-2 hours...



Samples:  35%|███▍      | 1244/3600 [3:28:57<4:32:40,  6.94s/it, H=1=622, H=0=622, Proc=13204]

In [ ]:
"""
=============================================================================
BLOCK 7: TRAIN/TEST SPLIT AND SAVE
=============================================================================
"""

print("\n" + "="*80)
print("BLOCK 7: TRAIN/TEST SPLIT")
print("="*80)

# Combine and shuffle
dataset = dataset_hall + dataset_non_hall
random.shuffle(dataset)

# 80/20 split
split_idx = int(0.8 * len(dataset))
train_data = dataset[:split_idx]
test_data = dataset[split_idx:]

# Save
with open('llama32_train.json', 'w') as f:
    json.dump(train_data, f, indent=2)
with open('llama32_test.json', 'w') as f:
    json.dump(test_data, f, indent=2)

print(f"\n✓ Data saved:")
print(f"  Train: {len(train_data)} samples")
print(f"  Test: {len(test_data)} samples")

# Class distribution
train_h0 = sum(1 for x in train_data if x['label'] == 0)
train_h1 = sum(1 for x in train_data if x['label'] == 1)
test_h0 = sum(1 for x in test_data if x['label'] == 0)
test_h1 = sum(1 for x in test_data if x['label'] == 1)

print(f"\n✓ Distribution:")
print(f"  Train: H=0={train_h0}, H=1={train_h1}")
print(f"  Test:  H=0={test_h0}, H=1={test_h1}")

# Example
ex = dataset[0]
print(f"\n✓ Example:")
print(f"  Label: {ex['label']}")
print(f"  Entity: {ex['entity']}")
print(f"  Embedding dim: {len(ex['embedding'])}")
print(f"  Text: {ex['text'][:80]}...")

# Free memory
del model, tokenizer
torch.cuda.empty_cache()
gc.collect()

print("\n" + "="*80)
print("BLOCKS 6-7 COMPLETE - DATA READY FOR TRAINING")
print("="*80)

In [ ]:
"""
=============================================================================
BLOCK 8: MLP CLASSIFIER (5 LAYERS - MIND PAPER)
=============================================================================
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import json
import random
import numpy as np
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             roc_auc_score, confusion_matrix, classification_report)
from tqdm import tqdm

print("="*80)
print("BLOCK 8: MLP CLASSIFIER DEFINITION")
print("="*80)

class MINDClassifier(nn.Module):
    """
    5-layer MLP following MIND paper Equation (1):
        P = MLP(ReLU(W · H + b))

    Input: H (1×n contextualized embedding from last token, last layer)
    Output: P (binary hallucination label)
    Loss: Binary Cross-Entropy (Equation 2)
    """

    def __init__(self, input_dim):
        super().__init__()

        self.layers = nn.Sequential(
            # Layer 1: input → 512
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),

            # Layer 2: 512 → 256
            nn.Linear(512, 256),
            nn.ReLU(),

            # Layer 3: 256 → 128
            nn.Linear(256, 128),
            nn.ReLU(),

            # Layer 4: 128 → 64
            nn.Linear(128, 64),
            nn.ReLU(),

            # Layer 5: 64 → 2 (output)
            nn.Linear(64, 2)
        )

    def forward(self, H):
        """
        Args:
            H: [batch_size, input_dim] - hidden states
        Returns:
            P: [batch_size, 2] - logits
        """
        return self.layers(H)


class MINDDataset(Dataset):
    def __init__(self, data):
        self.embeddings = torch.FloatTensor([d['embedding'] for d in data])
        self.labels = torch.LongTensor([d['label'] for d in data])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]


print("✓ MINDClassifier: 5 layers (Input → 512 → 256 → 128 → 64 → 2)")
print("✓ Activation: ReLU")
print("✓ Loss: Binary Cross-Entropy (Equation 2)")




In [ ]:
"""
=============================================================================
BLOCK 9: LOAD DATA AND TRAIN
=============================================================================
"""

print("\n" + "="*80)
print("BLOCK 9: TRAINING")
print("="*80)

# Load data
print("\n[Step 1] Loading data...")
with open('llama32_train.json', 'r') as f:
    train_data = json.load(f)
with open('llama32_test.json', 'r') as f:
    test_data = json.load(f)

print(f"✓ Train: {len(train_data)} samples")
print(f"✓ Test: {len(test_data)} samples")

# Create datasets
train_dataset = MINDDataset(train_data)
test_dataset = MINDDataset(test_data)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Create model
input_dim = len(train_data[0]['embedding'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MINDClassifier(input_dim).to(device)

print(f"\n[Step 2] Model setup:")
print(f"  Input dim: {input_dim}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Device: {device}")

# Training setup (MIND paper Section 5.3)
EPOCHS = 10
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-5

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss()

print(f"\n[Step 3] Training config:")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Weight decay: {WEIGHT_DECAY}")
print(f"  Optimizer: Adam")

# Training loop
print(f"\n[Step 4] Training...\n")
print("="*80)

best_test_acc = 0

for epoch in range(EPOCHS):
    # ═══════════════════════════════════════════════════════════════════
    # TRAINING
    # ═══════════════════════════════════════════════════════════════════
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for embeddings, labels in train_loader:
        embeddings = embeddings.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(embeddings)  # P = MLP(ReLU(W · H + b))
        loss = criterion(outputs, labels)  # L_BCE
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += labels.size(0)

    train_loss /= len(train_loader)
    train_acc = train_correct / train_total

    # ═══════════════════════════════════════════════════════════════════
    # TESTING
    # ═══════════════════════════════════════════════════════════════════
    model.eval()
    test_loss = 0
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for embeddings, labels in test_loader:
            embeddings = embeddings.to(device)
            labels = labels.to(device)

            outputs = model(embeddings)
            loss = criterion(outputs, labels)

            test_loss += loss.item()
            test_correct += (outputs.argmax(1) == labels).sum().item()
            test_total += labels.size(0)

    test_loss /= len(test_loader)
    test_acc = test_correct / test_total

    # Save best
    if test_acc > best_test_acc:
        best_test_acc = test_acc
        torch.save(model.state_dict(), 'llama32_mind_best.pth')
        star = " ★"
    else:
        star = ""

    print(f"Epoch {epoch+1:2d}/{EPOCHS}: "
          f"Train Loss={train_loss:.4f} Acc={train_acc:.4f} | "
          f"Test Loss={test_loss:.4f} Acc={test_acc:.4f}{star}")

print("="*80)
print(f"✓ Training complete! Best: {best_test_acc:.4f}")


In [ ]:
"""
=============================================================================
BLOCK 10: COMPREHENSIVE EVALUATION
=============================================================================
"""

print("\n" + "="*80)
print("BLOCK 10: COMPREHENSIVE EVALUATION")
print("="*80)

# Load best model
model.load_state_dict(torch.load('llama32_mind_best.pth'))
model.eval()

# Collect predictions
print("\n[Step 1] Computing predictions...")

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for embeddings, labels in test_loader:
        embeddings = embeddings.to(device)
        outputs = model(embeddings)
        probs = F.softmax(outputs, dim=1)

        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

# Calculate metrics
print("\n[Step 2] Computing metrics...")

accuracy = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')
auc = roc_auc_score(all_labels, all_probs)
cm = confusion_matrix(all_labels, all_preds)

# Print results
print("\n" + "="*80)
print("FINAL TEST RESULTS")
print("="*80)

print(f"\n{'Metric':<25} {'Value':<10} {'Interpretation':<40}")
print("-" * 75)
print(f"{'Accuracy':<25} {accuracy:.4f}     {int(accuracy*len(test_data))} of {len(test_data)} correct")
print(f"{'Precision':<25} {precision:.4f}     {int(precision*100):.0f}% of predicted hallucinations are real")
print(f"{'Recall':<25} {recall:.4f}     Catches {int(recall*100):.0f}% of actual hallucinations")
print(f"{'F1 Score':<25} {f1:.4f}     Harmonic mean of precision & recall")
print(f"{'AUC-ROC':<25} {auc:.4f}     Discrimination ability (>0.85 = excellent)")

print(f"\n{'Confusion Matrix:':<25}")
print(f"{'':>20} Predicted")
print(f"{'':>20} Non-H  Hall")
print(f"{'Actual Non-H':>20} {cm[0,0]:5d} {cm[0,1]:5d}  ({cm[0,1]} false alarms)")
print(f"{'Hall':>20} {cm[1,0]:5d} {cm[1,1]:5d}  ({cm[1,0]} missed)")

print(f"\n{'Detailed Counts:':<25}")
print(f"  True Negatives (TN):  {cm[0,0]:4d}  ← Correctly identified non-hallucinations")
print(f"  False Positives (FP): {cm[0,1]:4d}  ← Incorrectly flagged as hallucinations")
print(f"  False Negatives (FN): {cm[1,0]:4d}  ← Missed actual hallucinations (DANGER!)")
print(f"  True Positives (TP):  {cm[1,1]:4d}  ← Correctly detected hallucinations")

print(f"\n{'Per-Class Report:':<25}")
print(classification_report(
    all_labels, all_preds,
    target_names=['Non-hallucination (0)', 'Hallucination (1)'],
    digits=4
))

# Text examples
print("="*80)
print("TEXT EXAMPLES WITH PREDICTIONS")
print("="*80)

n_examples = min(20, len(test_data))
samples = random.sample(range(len(test_data)), n_examples)

print(f"\nShowing {n_examples} random test examples:\n")

correct_count = 0
for i, idx in enumerate(samples, 1):
    sample = test_data[idx]

    # Predict
    emb = torch.FloatTensor(sample['embedding']).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(emb)
        probs = F.softmax(out, dim=1)
        pred = out.argmax(1).item()
        conf = probs[0, pred].item()

    true = sample['label']
    is_correct = pred == true
    if is_correct:
        correct_count += 1

    check = "✓" if is_correct else "✗"

    print(f"[{i}] {check}")
    print(f"  True:       {['Non-hallucination', 'Hallucination'][true]}")
    print(f"  Predicted:  {['Non-hallucination', 'Hallucination'][pred]} ({conf:.1%} confident)")
    print(f"  Entity:     {sample['entity']}")
    print(f"  Text:       {sample['text'][:100]}...")
    print()

print(f"Examples summary: {correct_count}/{n_examples} correct ({100*correct_count/n_examples:.1f}%)")

# Final summary
print("\n" + "="*80)
print("ALL BLOCKS COMPLETE!")
print("="*80)
print(f"\nSummary:")
print(f"  Model: meta-llama/Llama-3.2-1B (4-bit)")
print(f"  Dataset: {len(train_data) + len(test_data)} samples")
print(f"  Test Accuracy: {accuracy:.4f}")
print(f"  Test AUC-ROC: {auc:.4f}")
print(f"  Test Precision: {precision:.4f}")
print(f"  Test Recall: {recall:.4f}")
print(f"  Test F1: {f1:.4f}")
print(f"\nFiles saved:")
print(f"  - llama32_train.json")
print(f"  - llama32_test.json")
print(f"  - llama32_mind_best.pth")
print("="*80)